# Julia notebook to compute the symbolic quadratures for the solution of the robust diagnostic PG problem
twnh 

Aug '26.  

This notebook reads the output from `BuoyancyEquationSolution.ipynb` and `FrictionalGeostrophicEquationSolution.ipynb`, namely the Green's functions $G_{\mathfrak{u}}$ and $G_b$ and computes integrals using them.

In [1]:
using SymPy
using Symbolics
using JLD2

# using Infiltrator

In [2]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                # Horizontal coordinates
# H      = SymFunction("H", real=true, positive=true) # Domain depth H(x,y)
H      = symbols("H",   real=true, positive=true)   # Domain depth
# HH     = symbols("HH",   real=true, positive=true)  # Domain depth H(x,y) evaluated at a specific point (x0,y0)

ψ      = symbols("ψ",     real=true, positive=true) 
φ      = symbols("φ", imaginary=true)

κ      = symbols("κ",     real=true, positive=true)  # Diffusivity parameter
ν      = symbols("ν",     real=true, positive=true)  # Viscosity parameters

# Dictionary for parsing strings.
locals = Dict(
    "x" => x,
    "y" => y,
    "z" => z,
    "ξ" => ξ,
    "κ" => κ,
    "ν" => ν,
    "φ" => φ,
    "ψ" => ψ,
    "H" => H
    )

# Parse expressions using the canonical symbols.
Guv_str = read("sympy_Guv_expression.txt", String)
Guv = sympy.sympify(Guv_str, locals=locals)
println("Guv expression loaded from file:")
display(Guv)
println()

Gb_str  = read("sympy_Gb_expression.txt", String)
Gb  = sympy.sympify(Gb_str,  locals=locals)
println("Gb expression loaded from file:")
display(Gb)
println()

Guv expression loaded from file:

Gb expression loaded from file:



⎧⎛ 2⋅ξ⋅φ    2⋅φ⋅(H + z)    2⋅φ⋅(H + z + ξ)    ⎞  -φ⋅(z + ξ)           
⎪⎝ℯ      - ℯ            - ℯ                + 1⎠⋅ℯ                     
⎪──────────────────────────────────────────────────────────  for z ≤ ξ
⎪                          ⎛ 2⋅H⋅φ    ⎞                               
⎪                    2⋅ν⋅φ⋅⎝ℯ      + 1⎠                               
⎨                                                                     
⎪⎛ 2⋅z⋅φ    2⋅φ⋅(H + ξ)    2⋅φ⋅(H + z + ξ)    ⎞  -φ⋅(z + ξ)           
⎪⎝ℯ      - ℯ            - ℯ                + 1⎠⋅ℯ                     
⎪──────────────────────────────────────────────────────────  for z ≥ ξ
⎪                          ⎛ 2⋅H⋅φ    ⎞                               
⎩                    2⋅ν⋅φ⋅⎝ℯ      + 1⎠                               

⎧⎛     -2⋅ξ⋅ψ⎞ ⎛ 2⋅H⋅ψ    -2⋅z⋅ψ⎞  ψ⋅(z + ξ)           
⎪⎝1 - ℯ      ⎠⋅⎝ℯ      + ℯ      ⎠⋅ℯ                    
⎪───────────────────────────────────────────  for z ≤ ξ
⎪                  ⎛ 2⋅H⋅ψ    ⎞                        
⎪            2⋅κ⋅ψ⋅⎝ℯ      + 1⎠                        
⎨                                                      
⎪⎛     -2⋅z⋅ψ⎞ ⎛ 2⋅H⋅ψ    -2⋅ξ⋅ψ⎞  ψ⋅(z + ξ)           
⎪⎝1 - ℯ      ⎠⋅⎝ℯ      + ℯ      ⎠⋅ℯ                    
⎪───────────────────────────────────────────  for z ≥ ξ
⎪                  ⎛ 2⋅H⋅ψ    ⎞                        
⎩            2⋅κ⋅ψ⋅⎝ℯ      + 1⎠                        

In [3]:
@time begin
    zp      = symbols("zp",     real=true, positive=true)
    # tmp = sympy.factor(sympy.integrate(Gb.subs(z, zp), (zp, ξ, 0)).args[1].args[1])
    tmp = sympy.factor(sympy.integrate(Gb.subs(z, zp), (zp, ξ, 0)))
end
println("1.1 Symbolic computation of the integral of Gb completed.")
display(tmp)

  0.256010 seconds (144.43 k allocations: 7.080 MiB, 15.05% compilation time)
1.1 Symbolic computation of the integral of Gb completed.


                               2         
 ⎛ 2⋅H⋅ψ  2⋅ξ⋅ψ    ⎞ ⎛ ξ⋅ψ    ⎞   -2⋅ξ⋅ψ 
-⎝ℯ     ⋅ℯ      + 1⎠⋅⎝ℯ    - 1⎠ ⋅ℯ       
─────────────────────────────────────────
                2 ⎛ 2⋅H⋅ψ    ⎞           
           2⋅κ⋅ψ ⋅⎝ℯ      + 1⎠           

In [ ]:
@time begin
    tmp2 = sympy.expand(Guv * tmp)
    # tmp3 = sympy.integrate(tmp2, (ξ, -H(x,y), 0))
    tmp3 = sympy.integrate(tmp2, (ξ, -H, 0))
    tmp3 = tmp3.args[1].args[1]
    # max_obj = sympy.Max(z, -H(x, y))
    max_obj = sympy.Max(-H, z)
    tmp4 = tmp3.subs(max_obj,z)
    tmp5 = sympy.factor(tmp4)
    tmp6 = sympy.simplify(tmp5)
end

println("1.2 Symbolic computation of the integral of G_u and Gb completed.")
display(tmp6)

In [ ]:
@time symbolics_expr = sympy_to_symbolics(tmp6, [])
display(symbolics_expr)

@save "symexpr.jld2" symbolics_expr
println("1.3 Expression converted to Symbolics and saved to file: symexpr.jld2")